# Logit Cloud — Predicción Local + Kaggle Submission

**Corre localmente** después de descargar el modelo entrenado en Kaggle Notebooks.

**Seleccionar el modelo a usar con `MODEL_NAME`:**

| `MODEL_NAME` | Archivos necesarios | Notebook origen |
|---|---|---|
| `'logit'` | `logit_cloud_best.pkl`, `logit_cloud_metadata.json` | `logit_cloud_train.ipynb` |
| `'logit_reg'` | `logit_reg_cloud_best.pkl`, `logit_reg_cloud_metadata.json` | `logit_reg_cloud_train.ipynb` |
| `'pygam'` | `pygam_cloud_best.pkl`, `logit_splines_cloud_metadata.json` | `logit_splines_cloud_train.ipynb` |

In [6]:
# ── SELECCIONAR MODELO ─────────────────────────────────────────────────────────
# Opciones: 'logit', 'logit_reg', 'pygam'
MODEL_NAME = 'logit'

In [7]:
import joblib
import pandas as pd
import numpy as np
import json
import time
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

print(f'MODEL_NAME seleccionado: {MODEL_NAME}')
print('Imports OK')

MODEL_NAME seleccionado: logit
Imports OK


In [8]:
MODEL_DIR = Path('.')
DATA_DIR  = Path('../../data/processed')

# Mapeo de MODEL_NAME a archivos
_configs = {
    'logit'    : ('logit_cloud_best.pkl',     'logit_cloud_metadata.json',
                  'submission_cloud_logit.csv',     'logit'),
    'logit_reg': ('logit_reg_cloud_best.pkl', 'logit_reg_cloud_metadata.json',
                  'submission_cloud_logit_reg.csv', 'logit_reg'),
    'pygam'    : ('pygam_cloud_best.pkl',     'logit_splines_cloud_metadata.json',
                  'submission_cloud_pygam.csv',     'pygam'),
}

assert MODEL_NAME in _configs, f'MODEL_NAME debe ser uno de: {list(_configs.keys())}'
model_file_name, meta_file_name, sub_file_name, label = _configs[MODEL_NAME]

model_file = MODEL_DIR / model_file_name
meta_file  = MODEL_DIR / meta_file_name

print(f'MODEL_DIR : {MODEL_DIR.resolve()}')
for f in [model_file, meta_file]:
    status = '✓ encontrado' if f.exists() else '✗ NO encontrado — descargar desde Kaggle'
    print(f'  {f.name} : {status}')

MODEL_DIR : C:\Users\HP\OneDrive\Escritorio\David Guzzi\Github\MECMT07\cloud\logit
  logit_cloud_best.pkl : ✓ encontrado
  logit_cloud_metadata.json : ✓ encontrado


In [9]:
with open(meta_file, 'r') as f:
    meta = json.load(f)

print(f'Metadata del modelo ({MODEL_NAME}):')
print(f'  Timestamp: {meta["timestamp"]}')

if MODEL_NAME == 'logit':
    _cv_auc_oof  = meta.get('cv_auc_oof', meta.get('best_cv_auc', 'N/A'))
    _cv_auc_mean = meta.get('cv_auc_mean', 'N/A')
    print(f'  CV AUC (5-fold OOF): {_cv_auc_oof}')
    print(f'  CV AUC (mean folds): {_cv_auc_mean}')
    print(f'  Train AUC          : {meta["train_auc"]}')
    print(f'  C                  : {meta["C"]}')
elif MODEL_NAME == 'logit_reg':
    print(f'  Mejor modelo   : {meta["best_model_name"]}')
    print(f'  CV AUC (5-fold): {meta["best_cv_auc"]}')
    print(f'  Train AUC      : {meta["train_auc"]}')
    print(f'  Mejor C        : {meta["best_C"]}')
elif MODEL_NAME == 'pygam':
    print(f'  CV AUC (5-fold): {meta["best_cv_auc"]}')
    print(f'  Train AUC      : {meta["train_auc"]}')
    print(f'  Lambda óptimo  : {meta["best_lam"]}')
    print(f'  n_folds        : {meta.get("n_folds", 5)}')
    print(f'  n_splines      : {meta.get("n_splines", 10)}')

feature_cols = meta['feature_cols']
print(f'  n_features     : {len(feature_cols)}')

Metadata del modelo (logit):
  Timestamp: 2026-02-25T19:11:31.217137
  CV AUC (5-fold OOF): 0.749695
  CV AUC (mean folds): 0.749736
  Train AUC          : 0.749991
  C                  : 1000000.0
  n_features     : 30


In [10]:
print(f'Cargando modelo: {model_file.name}  ({model_file.stat().st_size/1e6:.2f} MB)...')
bundle = joblib.load(model_file)
print('Bundle cargado ✓')
print(f'  Claves en bundle: {list(bundle.keys())}')

Cargando modelo: logit_cloud_best.pkl  (0.00 MB)...
Bundle cargado ✓
  Claves en bundle: ['pipeline', 'feature_cols']


In [11]:
print('Cargando features_test.parquet...')
df_test     = pd.read_parquet(DATA_DIR / 'features_test.parquet')
sk_ids_test = df_test['SK_ID_CURR'].values

# Encodear categóricas si existen
cat_cols = [c for c in feature_cols if df_test[c].dtype == 'object']
if cat_cols:
    for col in cat_cols:
        cats    = df_test[col].dropna().unique()
        mapping = {v: i for i, v in enumerate(cats)}
        df_test[col] = df_test[col].map(mapping)

X_test_raw = df_test[feature_cols].values
print(f'  Test shape  : {X_test_raw.shape}')
print(f'  Features    : {len(feature_cols)}')

Cargando features_test.parquet...
  Test shape  : (48744, 30)
  Features    : 30


In [12]:
print(f'Generando predicciones con modelo: {MODEL_NAME}...')

if MODEL_NAME == 'logit':
    # Pipeline sklearn (imputer + scaler + lr) todo incluido
    pipeline     = bundle['pipeline']
    y_test_prob  = pipeline.predict_proba(X_test_raw)[:, 1]

elif MODEL_NAME == 'logit_reg':
    # Modelo LogisticRegressionCV + imputer + scaler separados
    model   = bundle['model']
    imputer = bundle['imputer']
    scaler  = bundle['scaler']
    X_imp   = imputer.transform(X_test_raw)
    X_sc    = scaler.transform(X_imp)
    y_test_prob = model.predict_proba(X_sc)[:, 1]

elif MODEL_NAME == 'spline_logit':
    # Pipeline SplineTransformer + logit, más imputer separado
    pipeline = bundle['pipeline']
    imputer  = bundle['imputer']
    X_imp    = imputer.transform(X_test_raw)
    y_test_prob = pipeline.predict_proba(X_imp)[:, 1]

elif MODEL_NAME == 'pygam':
    # GAM + scaler + imputer
    gam     = bundle['gam']
    scaler  = bundle['scaler']
    imputer = bundle['imputer']
    X_imp   = imputer.transform(X_test_raw)
    X_sc    = scaler.transform(X_imp)
    y_test_prob = gam.predict_proba(X_sc)

print(f'  Predicciones generadas : {len(y_test_prob):,}')
print(f'  Score mínimo           : {y_test_prob.min():.5f}')
print(f'  Score máximo           : {y_test_prob.max():.5f}')
print(f'  Score medio            : {y_test_prob.mean():.5f}')
print(f'  % predicho como default: {(y_test_prob > 0.5).mean()*100:.2f}%')

Generando predicciones con modelo: logit...
  Predicciones generadas : 48,744
  Score mínimo           : 0.00000
  Score máximo           : 0.99365
  Score medio            : 0.42569
  % predicho como default: 34.32%


In [13]:
submission = pd.DataFrame({'SK_ID_CURR': sk_ids_test, 'TARGET': y_test_prob})
out_path   = DATA_DIR / sub_file_name
submission.to_csv(out_path, index=False)

print(f'Submission guardado: {out_path}')
print(f'Shape: {submission.shape}')
display(submission.head())

Submission guardado: ..\..\data\processed\submission_cloud_logit.csv
Shape: (48744, 2)


,SK_ID_CURR,TARGET
0,100001,0.394567
1,100005,0.678356
2,100013,0.300972
3,100028,0.269377
4,100038,0.606621


## Kaggle Submission — AUC Test (Public Leaderboard)

> **Límite**: 5 submissions/día.

In [14]:
from kaggle import KaggleApi

COMPETITION = 'home-credit-default-risk'

# Construir mensaje descriptivo según modelo + extraer _cv_auc para el RESULTADO block
if MODEL_NAME == 'logit':
    _cv_auc = meta.get('cv_auc_oof', meta.get('best_cv_auc', 0))
    _msg = f"Logit Cloud | CV AUC={_cv_auc:.5f} | train AUC={meta['train_auc']:.5f}"
elif MODEL_NAME == 'logit_reg':
    _cv_auc = meta.get('best_cv_auc', 0)
    _msg = f"Logit Reg Cloud | {meta['best_model_name']} | CV AUC={_cv_auc:.5f} | train AUC={meta['train_auc']:.5f}"
elif MODEL_NAME == 'pygam':
    _cv_auc = meta['best_cv_auc']
    _msg = f"pyGAM Cloud | lam={meta['best_lam']:.5f} | CV AUC={_cv_auc:.5f} | train AUC={meta['train_auc']:.5f}"

def _as_str(x):
    return '' if x is None else str(x)

def _get_score(api, comp, file_name, message, poll=10, timeout=300):
    start = time.time()
    while time.time() - start < timeout:
        subs = api.competition_submissions(comp)
        matched = next(
            (s for s in subs
             if _as_str(getattr(s, 'file_name', None)) == file_name
             and _as_str(getattr(s, 'description', None)) == message),
            subs[0] if subs else None
        )
        if matched:
            pub    = _as_str(getattr(matched, 'public_score',  None))
            status = _as_str(getattr(matched, 'status',        None))
            elapsed = int(time.time() - start)
            print(f'  [{elapsed:>3}s] status={status!r}  public_score={pub!r}')
            if pub and pub.lower() not in ('', 'none', 'null', '-'):
                priv = _as_str(getattr(matched, 'private_score', None))
                return float(pub), (float(priv) if priv and priv.lower() not in ('','none','null','-') else None)
        time.sleep(poll)
    return None, None

_api = KaggleApi()
_api.authenticate()

print(f'Enviando: {_msg}')
_api.competition_submit(file_name=str(out_path), message=_msg, competition=COMPETITION)
print('Esperando scoring (poll 10 s / máx 5 min)...')

public_auc, private_auc = _get_score(_api, COMPETITION, out_path.name, _msg)

print(f'\n' + '=' * 65)
print(f'RESULTADO KAGGLE — {MODEL_NAME.upper()} CLOUD')
print(f'=' * 65)
print(f'  AUC test Public  LB  : {public_auc}')
print(f'  AUC test Private LB  : {private_auc}')
print(f'  CV AUC (entrenamiento): {_cv_auc:.5f}')
if public_auc:
    print(f'  Gap CV - Public LB   : {abs(_cv_auc - public_auc):.5f}')
print(f'=' * 65)

Enviando: Logit Cloud | CV AUC=0.74970 | train AUC=0.74999


100%|██████████| 1.27M/1.27M [00:01<00:00, 796kB/s] 


Esperando scoring (poll 10 s / máx 5 min)...
  [  0s] status='SubmissionStatus.PENDING'  public_score=''
  [ 10s] status='SubmissionStatus.COMPLETE'  public_score='0.74983'

RESULTADO KAGGLE — LOGIT CLOUD
  AUC test Public  LB  : 0.74983
  AUC test Private LB  : 0.73516
  CV AUC (entrenamiento): 0.74970
  Gap CV - Public LB   : 0.00013
